In [ ]:
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
from tqdm import tqdm

In [ ]:
pd.set_option('display.max_colwidth', 200)

In [ ]:
service = Service(r"C:\chromedriver.exe")
driver = webdriver.Chrome(service=service)

url = 'https://www.citilink.ru/'

categories_dict = {}
categories, category_urls = [], []

try:
    driver.get(url)
    time.sleep(10)
    try:
        categories_container = driver.find_element(By.XPATH, "//div[@data-meta-name='CategoryTilesLayout__category-tiles']")
        category_links = categories_container.find_elements(By.TAG_NAME, "a")

        category_urls = []
        categories = []
        for link in category_links:
            category_name = link.text
            url = link.get_attribute("href")

            if url:
                category_urls.append(url)
            if category_name:
                categories.append(category_name)

        for cat, url in zip(categories, category_urls):
            categories_dict[cat] = url

    except Exception as e:
        print(f"Ошибка: Не найден контейнер категорий {e}")

finally:
    driver.quit()

In [ ]:
def get_product(url):
    service = Service(r"C:\chromedriver.exe")
    driver = webdriver.Chrome(service=service)

    product_urls = []
    product_names = []
    try:
        driver.get(url)
        time.sleep(10)

        try:
            products = driver.find_elements(By.XPATH, "//a[contains(@href, '/product/')]")

            for link in products:
                product_name = link.get_attribute('title')
                product_url = link.get_attribute('href')

                if product_url and product_name:
                    product_urls.append(product_url)
                    product_names.append(product_name)

            return product_urls, product_names

        except Exception as e:
            print(f"Ошибка: Не найден контейнер категорий {e}")

    finally:
        driver.quit()

In [ ]:
product_categories, caterory_urls, product_names, product_urls = [], [], [], []

for key, val in categories_dict.items():
    urls, names = get_product(val)

    for name, url in zip(names, urls):
        product_categories.append(key)
        caterory_urls.append(val)
        product_names.append(name)
        product_urls.append(url)

In [ ]:
data = pd.DataFrame({'category': product_categories,
                     'caterory_url': caterory_urls,
                     'product_name': product_names,
                     'product_url': product_urls
                    }
                   )

In [ ]:
data = data[~data['product_url'].str.contains('otzyvy')]
data.drop_duplicates(inplace = True)
data.reset_index(inplace = True,  drop = True)

In [ ]:
def get_reviews(product_url):
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-notifications")
    chrome_options.add_argument("--mute-audio")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    service = Service(r"C:\chromedriver.exe")
    driver = webdriver.Chrome(service=service, options=chrome_options)

    product_urls = []
    reviews_advantages, reviews_disadvantages, reviews_comments = [], [], []

    try:
        driver.get(product_url)
        time.sleep(3)

        try:
            reviews_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, "//a[contains(@href, 'otzyvy')]"))
            )
            driver.execute_script("arguments[0].click();", reviews_button)
            time.sleep(3)

            c = 0
            while True:
                try:
                    show_more_button = WebDriverWait(driver, 5).until(
                        EC.element_to_be_clickable((By.XPATH, "//button[contains(@class, 'ekx3zbi0') and span[text()='Показать ещё']]"))
                    )
                    driver.execute_script("arguments[0].click();", show_more_button)
                    time.sleep(3)
                    c += 1

                except Exception as e:
                    print(f"Кнопка 'Показать ещё' больше не доступна. Нажималась {c} раз")
                    break

            advantages = driver.find_elements(By.XPATH, "//h5[text()='Достоинства']/following-sibling::span")
            disadvantages = driver.find_elements(By.XPATH, "//h5[text()='Недостатки']/following-sibling::span")
            comments = driver.find_elements(By.XPATH, "//h5[text()='Комментарий']/following-sibling::span")

            max_length = max(len(advantages), len(disadvantages), len(comments))
            for i in range(max_length):
                adv_text = advantages[i].text if i < len(advantages) else None
                dis_text = disadvantages[i].text if i < len(disadvantages) else None
                com_text = comments[i].text if i < len(comments) else None

                reviews_advantages.append(adv_text)
                reviews_disadvantages.append(dis_text)
                reviews_comments.append(com_text)

            return [product_url] * max_length, reviews_advantages, reviews_disadvantages, reviews_comments

        except Exception as e:
            print(f"Ошибка при извлечении отзывов: {e}")
            return product_urls, reviews_advantages, reviews_disadvantages, reviews_comments

    finally:
        driver.quit()


In [ ]:
prod_urls, advantages, disadvantages, comments = [], [], [], []

for urls in tqdm(data['product_url']):
    urls_product, reviews_advantages, reviews_disadvantages, reviews_comments = get_reviews(urls)

    for url, adv, dis, com in zip(urls_product, reviews_advantages, reviews_disadvantages, reviews_comments):
        prod_urls.append(url)
        advantages.append(adv)
        disadvantages.append(dis)
        comments.append(com)

In [ ]:
data_reviews = pd.DataFrame({'product_url': prod_urls,
                             'advantages': advantages,
                             'disadvantages': disadvantages,
                             'comments': comments
                            }
                           )
data_reviews

In [ ]:
data_reviews['product_url'].nunique()

In [ ]:
data_reviews.groupby('product_url')['comments'].count().sort_values(ascending = False)

In [ ]:
df = pd.merge(data,
              data_reviews,
              how = 'left',
              on = 'product_url'
             )
df

In [ ]:
df.groupby('product_url')['comments'].count().sort_values(ascending = False)

In [ ]:
df.to_excel("Парсинг отзывов сайта ситилинк.xlsx",
            index = False
           )